# Lab 5: Playground concentration grid

This notebook summarizes the existing 1,008 playground points into 1 km squares. It keeps populated squares only and writes a GeoJSON layer for the Leaflet map.


In [1]:
from pathlib import Path

import geopandas as gpd
from pandas.api.types import is_numeric_dtype
from shapely.geometry import box

project_dir = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "A2_Yousif")
        if (candidate / "data" / "playgrounds.geojson").exists()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate data/playgrounds.geojson from the current folder")

input_path = project_dir / "data" / "playgrounds.geojson"
output_path = project_dir / "data" / "playground_grid.geojson"

playgrounds = gpd.read_file(input_path)
original_point_count = len(playgrounds)
assert original_point_count == 1008, f"Expected 1,008 playground points, found {original_point_count}"
assert set(playgrounds.geom_type) == {"Point"}, "The source data must contain points"
playgrounds = playgrounds.set_crs("EPSG:4326", allow_override=True)
playgrounds_metric = playgrounds.to_crs("EPSG:2950")

cell_size = 1000
min_x, min_y, max_x, max_y = playgrounds_metric.total_bounds
origin_x = (min_x // cell_size) * cell_size
origin_y = (min_y // cell_size) * cell_size
last_column = int((max_x - origin_x) // cell_size)
last_row = int((max_y - origin_y) // cell_size)

grid_records = []
for row in range(last_row + 1):
    for column in range(last_column + 1):
        cell_id = row * (last_column + 1) + column
        left = origin_x + column * cell_size
        bottom = origin_y + row * cell_size
        grid_records.append({
            "cell_id": cell_id,
            "geometry": box(left, bottom, left + cell_size, bottom + cell_size),
        })

grid_metric = gpd.GeoDataFrame(grid_records, geometry="geometry", crs="EPSG:2950")
playgrounds_metric = playgrounds_metric.reset_index(drop=True)
playgrounds_metric["point_id"] = playgrounds_metric.index

# Intersections include boundary points; sorting and keeping one match assigns each point once.
matches = gpd.sjoin(playgrounds_metric, grid_metric, how="left", predicate="intersects")
matches = matches.sort_values(["point_id", "cell_id"])
one_cell_per_point = matches.drop_duplicates(subset="point_id", keep="first")
counts = one_cell_per_point.groupby("cell_id").size().rename("playground_count")
grid_with_counts = grid_metric.merge(counts, on="cell_id", how="inner")
grid_with_counts["playground_count"] = grid_with_counts["playground_count"].astype("int64")
grid_wgs84 = grid_with_counts[["playground_count", "geometry"]].to_crs("EPSG:4326")

print("Playground count distribution:")
print(grid_wgs84["playground_count"].describe())
print("\nCounts by value:")
print(grid_wgs84["playground_count"].value_counts().sort_index().to_string())
print(f"\nMinimum: {grid_wgs84['playground_count'].min()}")
print(f"Maximum: {grid_wgs84['playground_count'].max()}")


Playground count distribution:
count    287.000000
mean       3.512195
std        2.058334
min        1.000000
25%        2.000000
50%        3.000000
75%        5.000000
max       12.000000
Name: playground_count, dtype: float64

Counts by value:
playground_count
1     31
2     96
3     31
4     53
5     24
6     32
7      7
8      6
9      3
10     2
12     2

Minimum: 1
Maximum: 12


## Class breaks

The observed counts range from 1 to 12 playgrounds per populated square. Five easy-to-explain classes are 1–2, 3–4, 5–6, 7–8, and 9–12. The wider last class groups the few high-count squares while preserving a light-to-dark order across the full observed range.


In [2]:
assert original_point_count == 1008
assert int(grid_wgs84["playground_count"].sum()) == original_point_count
assert is_numeric_dtype(grid_wgs84["playground_count"])
assert set(grid_wgs84.geom_type) == {"Polygon"}
assert grid_wgs84.crs.to_epsg() == 4326
assert (grid_wgs84["playground_count"] >= 0).all()

grid_wgs84.to_file(output_path, driver="GeoJSON")
assert output_path.is_file(), f"Output file was not created: {output_path}"

written_grid = gpd.read_file(output_path)
assert len(written_grid) == len(grid_wgs84)
assert written_grid.crs.to_epsg() == 4326
assert int(written_grid["playground_count"].sum()) == 1008
assert is_numeric_dtype(written_grid["playground_count"])
assert set(written_grid.geom_type) == {"Polygon"}
assert (written_grid["playground_count"] >= 0).all()

print(f"Playgrounds counted: {int(grid_wgs84['playground_count'].sum())} of {original_point_count}")
print("All checks passed")


Playgrounds counted: 1008 of 1008
All checks passed
